In [ ]:
import boto3

Get Information on all the EC2 instances available

In [ ]:
ec2 = boto3.client('ec2')
ec2.describe_instances()

# Create a new Key Pair for EC2 Instances

Shows how to generate a key pair used to identify at the EC2 instance. Key pair is saved to 'KeyMaterial' which basically contains the information of the pem file used for verification.

In [ ]:
resp = ec2.create_key_pair(KeyName='mhofmann')
resp

In [ ]:
file = open("creds/mhofmann.pem", "w")
file.write(resp['KeyMaterial'])
file.close()

# Create new EC2 Instance

Use this code to create a new EC2 instance

In [ ]:
response = ec2.run_instances(ImageId='ami-09278528675a8d54e', MinCount=1, MaxCount=1,
                            InstanceType='t3.micro', KeyName='mhofmann', BlockDeviceMappings=[{"DeviceName": "/dev/xvda", "Ebs": {"DeleteOnTermination": True,"VolumeSize": 20}}])

In [ ]:
response

Describe instances contains now two EC2 instances because one was available earlier and one was created now.

In [ ]:
ec2.describe_instances()

# Create Security Group

The new EC2 instance got now security groups assigned. They define who in general can access the machine and who not.\
Get the security groups with the command down below:

In [ ]:
response = ec2.describe_security_groups()

In [ ]:
response

Create a new empty security group, contains no security policies.

In [ ]:
response = ec2.create_security_group(GroupName='mhofmann', Description='Security Group for mhofmann testing')

In [ ]:
response

In [ ]:
security_group_id = response['GroupId']

New rules can be added to the security group.\
The rule down below allows all traffic to pass through.

In [ ]:
response = ec2.authorize_security_group_ingress(
    GroupId=security_group_id,IpPermissions=[{"IpProtocol": "tcp", "FromPort": 22, "ToPort": 22, "IpRanges": [{"CidrIp": "0.0.0.0/0"}]}])

In [ ]:
response

In [ ]:
response = ec2.describe_instances()

In [ ]:
instance_id = response["Reservations"][1]["Instances"][0]["InstanceId"]

In [ ]:
instance_id, security_group_id

In [ ]:
old_gid = response["Reservations"][1]["Instances"][0]["SecurityGroups"][0]["GroupId"]

In [ ]:
old_gid

The security group itself has to be added to the EC2 instance to make it available from the outside. It is done with the function below:

In [ ]:
ec2.modify_instance_attribute(InstanceId=instance_id, Groups=[security_group_id, old_gid])

# Managing the status of an instance

Here, some valuable functions for managing and monitoring the status of the EC2 are listed

In [ ]:
import time

def wait_for_status(instance_id, target_status):
    while True:
        response = ec2.describe_instances(InstanceIds=[instance_id])
        status = response['Reservations'][0]['Instances'][0]['State']['Name']
        if status == target_status:
            print(f"Instance {instance_id} is now in '{target_status}' state.")
            break
        else:
            print(f"Instance {instance_id} is still in status '{status}'. Waiting for status '{target_status}'...")
        time.sleep(10)  # check every 10 seconds

def start_instance(instance_id):
    ec2.start_instances(InstanceIds=[instance_id])
    print(f"Starting instance {instance_id}...")
    wait_for_status(instance_id, 'running')

In [ ]:
start_instance("i-0639ab58d8c53be27")

In [ ]:
def stop_instance(instance_id):
    ec2.stop_instances(InstanceIds=[instance_id])
    print(f"Stopping instance {instance_id}...")
    wait_for_status(instance_id, 'stopped')

In [ ]:
stop_instance("i-0639ab58d8c53be27")

In [ ]:
def terminate_instance(instance_id):
    ec2.terminate_instances(InstanceIds=[instance_id])
    print(f"Terminating instance {instance_id}...")
    wait_for_status(instance_id, 'terminated')

In [ ]:
terminate_instance("i-0639ab58d8c53be27")